# Korean to English Translation and Evaluation
이 노트북은 `backend-server/history.csv` 파일의 요약 결과 및 정답(ollama)을 영어로 번역하고, 번역된 결과를 바탕으로 ROUGE 및 BERTScore를 계산합니다. 번역은 로컬 Ollama를 통해 `gemma3:27b` 모델을 사용합니다.

In [1]:
import pandas as pd
import requests
import json
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import torch
import os

# tqdm pandas 연동
tqdm.pandas()

In [2]:
# CSV 파일 불러오기
input_path = 'backend-server/history.csv'
df = pd.read_csv(input_path)
print(f"데이터 크기: {df.shape}")
display(df.head())

데이터 크기: (28, 10)


,id,timestamp,title,context,tf_idf,text_rank,lsa,lex_rank,mmr,ollama
0,20260223141044,2026-02-23 14:10:44,"'제미나이 3.1 프로', 수능 테스트에서 최초로 만점 도달",구글의 최신 모델 ‘제미나이 3.1 프로’가 수능 만점에 도달한 최초의 AI 모델이...,구유겸 순천향대학교 컴퓨터소프트웨어공학과 학생은 자체 구축한 시스템을 활용해 ‘제미...,구글의 최신 모델 ‘제미나이 3.1 프로’가 수능 만점에 도달한 최초의 AI 모델이...,구글의 최신 모델 ‘제미나이 3.1 프로’가 수능 만점에 도달한 최초의 AI 모델이...,구글의 최신 모델 ‘제미나이 3.1 프로’가 수능 만점에 도달한 최초의 AI 모델이...,구유겸 순천향대학교 컴퓨터소프트웨어공학과 학생은 자체 구축한 시스템을 활용해 ‘제미...,구글의 제미나이 3.1 프로가 학생 구유겸의 자체 시스템을 통해 수능에서 만점을 받...
1,20260223141059,2026-02-23 14:10:59,"KAIST, 분자 에너지 구조 예측 AI 개발…“신약·신소재 개발에 도움”","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...","한국과학기술원(KAIST, 총장 이광형)은 김우연 교수 연구팀이 분자 안정성을 좌우...",한국과학기술원 연구팀은 분자 안정성을 예측하고 구조를 설계하는 AI 모델 '리만 확...
2,20260223141115,2026-02-23 14:11:15,"구글, 논문 일러스트 생성 AI ‘페이퍼바나나’ 공개...'환각 낮추고 미적 기준 올려'","AI가 참고 문헌 검색과 정리를 넘어, 학술 논문에 들어가는 이미지까지 처리하게 됐...",구글은 베이징대학교 연구진과 6일(현지시간) 출판 수준의 학술 도식과 그래프를 자동...,페이퍼바나나는 구글의 '나노바나나 프로' 등 최신 비전-언어 모델(VLM)과 이미지...,페이퍼바나나는 구글의 '나노바나나 프로' 등 최신 비전-언어 모델(VLM)과 이미지...,페이퍼바나나는 구글의 '나노바나나 프로' 등 최신 비전-언어 모델(VLM)과 이미지...,구글은 베이징대학교 연구진과 6일(현지시간) 출판 수준의 학술 도식과 그래프를 자동...,구글 연구진은 학술 논문의 도식과 그래프를 자동으로 생성하는 AI 프레임워크 ‘페이...
3,20260223141136,2026-02-23 14:11:36,"구글, 연구 에이전트 ‘마스’ 공개...'스스로 ‘교훈’ 얻어 실험 전략 수정'","AI가 논문을 읽고 코드를 작성하는 수준을 넘어, 이제는 연구 설계와 실험 전략까지...",구글과 스탠포드대학교 연구진은 4일(현지시간) AI 연구 전 과정을 자율적으로 수행...,"AI가 논문을 읽고 코드를 작성하는 수준을 넘어, 이제는 연구 설계와 실험 전략까지...","AI가 논문을 읽고 코드를 작성하는 수준을 넘어, 이제는 연구 설계와 실험 전략까지...","AI가 논문을 읽고 코드를 작성하는 수준을 넘어, 이제는 연구 설계와 실험 전략까지...","AI가 논문을 읽고 코드를 작성하는 수준을 넘어, 이제는 연구 설계와 실험 전략까지...",구글과 스탠포드대학교 연구진이 AI 연구 전 과정을 자율적으로 수행하는 에이전트 '...
4,20260223141152,2026-02-23 14:11:52,"KAIST, AI 반도체 추론 속도 높이는 ‘오토 GNN’ 개발","한국과학기술원(KAIST, 총장 이광형)은 정명수 전기및전자공학부 교수 연구팀이 그...","한국과학기술원(KAIST, 총장 이광형)은 정명수 전기및전자공학부 교수 연구팀이 그...","AI가 분석해야 하는 ‘데이터의 연결 방식’의 맞춰서, 반도체가 스스로 가장 효율적...","AI가 분석해야 하는 ‘데이터의 연결 방식’의 맞춰서, 반도체가 스스로 가장 효율적...","한국과학기술원(KAIST, 총장 이광형)은 정명수 전기및전자공학부 교수 연구팀이 그...","한국과학기술원(KAIST, 총장 이광형)은 정명수 전기및전자공학부 교수 연구팀이 그...",한국과학기술원 연구팀은 그래프 신경망 기반의 AI 추론 속도 향상 기술인 ‘오토GN...


In [3]:
# Ollama 번역 함수 정의
OLLAMA_API_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "gemma3:27b"

def translate_to_english(text):
    if pd.isna(text) or not str(text).strip():
        return text
    
    prompt = f"Translate the following Korean text to English. Output ONLY the translated English text, without any additional explanations, quotes, or greetings.\n\n[Korean Text]:\n{text}"
    
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False
    }
    
    try:
        response = requests.post(OLLAMA_API_URL, json=payload)
        response.raise_for_status()
        result = response.json()
        return result.get('response', '').strip()
    except Exception as e:
        print(f"Error translating text: {e}")
        return text

In [4]:
# 번역 진행 (대상 컬럼들 추출)
columns_to_translate = ['tf_idf', 'text_rank', 'lsa', 'lex_rank', 'mmr', 'ollama']

df_english = df.copy()

print("Starting translation...")
for col in columns_to_translate:
    if col in df_english.columns:
        print(f"Translating column: {col}")
        df_english[col] = df_english[col].progress_apply(translate_to_english)

# 번역본 저장
output_path = 'english_history.csv'
df_english.to_csv(output_path, index=False, encoding='utf-8')
print(f"Saved translated data to {output_path}")

Starting translation...
Translating column: tf_idf


  0%|          | 0/28 [00:00<?, ?it/s]

Translating column: text_rank


  0%|          | 0/28 [00:00<?, ?it/s]

Translating column: lsa


...

Translating column: mmr


  0%|          | 0/28 [00:00<?, ?it/s]

Translating column: ollama


  0%|          | 0/28 [00:00<?, ?it/s]

Saved translated data to english_history.csv


In [5]:
# 이미 번역된 파일이 있다면 아래 코드로 바로 불러올 수 있습니다.
# df_english = pd.read_csv('english_history.csv')

# 평가지표 계산 준비
methods = ['tf_idf', 'text_rank', 'lsa', 'lex_rank', 'mmr']
ground_truth_col = 'ollama'

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
results = []

# ROUGE 점수 먼저 계산하며 데이터 정리
for index, row in tqdm(df_english.iterrows(), total=len(df_english), desc="Calculating ROUGE"):
    target = str(row[ground_truth_col])
    
    for method in methods:
        if method not in row:
            continue
        
        pred = str(row[method])
        
        if pd.isna(pred) or not pred.strip() or pd.isna(target) or not target.strip():
             continue
        
        rouge_scores = scorer.score(target, pred)
        
        results.append({
            'id': row['id'],
            'method': method,
            'prediction': pred,
            'target': target,
            'rouge1': rouge_scores['rouge1'].fmeasure,
            'rouge2': rouge_scores['rouge2'].fmeasure,
            'rougeL': rouge_scores['rougeL'].fmeasure
        })

results_df = pd.DataFrame(results)

Calculating ROUGE:   0%|          | 0/28 [00:00<?, ?it/s]

In [6]:
# BERTScore 계산
# 주의: bert_score 실행 시 모델 다운로드가 진행될 수 있습니다. (예: roberta-large)
print("Calculating BERTScore...")
predictions = results_df['prediction'].tolist()
targets = results_df['target'].tolist()

batch_size = 16
bert_p, bert_r, bert_f1 = [], [], []

for i in tqdm(range(0, len(predictions), batch_size), desc="BERTScore Batch"):
    batch_preds = predictions[i:i+batch_size]
    batch_targets = targets[i:i+batch_size]
    
    # lang="en"으로 설정하여 영문 BERTScore 계산
    P, R, F1 = bert_score(batch_preds, batch_targets, lang="en", verbose=False)
    
    bert_p.extend(P.tolist())
    bert_r.extend(R.tolist())
    bert_f1.extend(F1.tolist())

results_df['bert_precision'] = bert_p
results_df['bert_recall'] = bert_r
results_df['bert_f1'] = bert_f1

Calculating BERTScore...


BERTScore Batch:   0%|          | 0/9 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

In [7]:
# 점수 요약 및 저장
summary_scores = results_df.groupby('method').agg({
    'rouge1': 'mean',
    'rouge2': 'mean',
    'rougeL': 'mean',
    'bert_precision': 'mean',
    'bert_recall': 'mean',
    'bert_f1': 'mean'
}).reset_index()

print("============= 평가 지표 요약 =============")
display(summary_scores)

# CSV 저장
results_df.drop(columns=['prediction', 'target']).to_csv('english_scores_detailed.csv', index=False)
summary_scores.to_csv('english_scores_summary.csv', index=False)
print("Scoring complete. Results saved to english_scores_detailed.csv and english_scores_summary.csv")

============= 평가 지표 요약 =============


,method,rouge1,rouge2,rougeL,bert_precision,bert_recall,bert_f1
0,lex_rank,0.496020,0.193609,0.293890,0.879778,0.888729,0.884154
1,lsa,0.482153,0.191961,0.284601,0.878274,0.887634,0.882867
2,mmr,0.513033,0.216503,0.299766,0.869260,0.900763,0.884691
3,text_rank,0.507980,0.205133,0.300635,0.879827,0.892799,0.886197
4,tf_idf,0.513817,0.221813,0.303127,0.870346,0.901821,0.885765


Scoring complete. Results saved to english_scores_detailed.csv and english_scores_summary.csv
